# SplatStream Lab — 3DGS Training Efficiency Study
Netflix JR40251 extension: training time vs held-out quality.

Run the three-preset pilot first. No performance claim should be made until the measured JSON is reviewed.

In [ ]:
from pathlib import Path
import subprocess

repo = Path('/content/splatstream-lab')
if not repo.exists():
    subprocess.run(['git','clone','--branch','fastgs-training-efficiency-study','https://github.com/reusahn/splatstream-lab.git',str(repo)], check=True)
else:
    subprocess.run(['git','fetch','origin'], cwd=repo, check=True)
    subprocess.run(['git','checkout','fastgs-training-efficiency-study'], cwd=repo, check=True)
    subprocess.run(['git','pull','--ff-only'], cwd=repo, check=True)
print(subprocess.check_output(['git','rev-parse','HEAD'], cwd=repo, text=True).strip())

In [ ]:
import runpy
setup = runpy.run_path('/content/splatstream-lab/tools/run_bonsai_cuda.py')
setup['check_environment']()
setup['setup_gsplat']()
setup['prepare_dataset']()

## Pilot
Baseline 7K, pure 5K early stop, and 5K with densification stopped at 3K.

Run these on the same GPU/runtime so the training-time comparison is meaningful.

In [ ]:
import subprocess
cmd = [
    'python', '/content/splatstream-lab/tools/run_training_efficiency_sweep.py',
    '--gsplat-dir', '/content/gsplat',
    '--data-dir', '/content/gsplat/examples/data/360_v2/bonsai',
    '--config', '/content/splatstream-lab/configs/training_efficiency_sweep.json',
    '--only', 'baseline_7k', 'early_stop_5k', 'densify_stop_3k_5k'
]
subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path
import json
result = Path('/content/gsplat/examples/results/splatstream_training_efficiency/training_efficiency_results.json')
data = json.loads(result.read_text())
for run in data['presets']:
    p = run['preset']
    print(p['name'], 'wall=', run.get('wall_seconds_external'), 'GS=', run.get('gaussians'), 'PSNR=', run.get('psnr'), 'SSIM=', run.get('ssim'), 'LPIPS=', run.get('lpips'))

In [ ]:
from google.colab import files
files.download('/content/gsplat/examples/results/splatstream_training_efficiency/training_efficiency_results.json')